# WattWise AI - Step 3: Multi-Step Forecasting & Bill Estimation
This notebook details the Random Forest autoregressive forecasting architecture, uncertainty quantification, naive benchmark evaluation, and tariff bill estimation.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.forecast import make_forecast_features, train_rf, predict, evaluate_forecasting
from src.bill import estimate_bill

## 1. Load Hourly Dataset & Evaluate Forecaster

In [ ]:
df_hourly = pd.read_parquet('../data/processed/hourly.parquet')
metrics_df, details = evaluate_forecasting(df_hourly, test_days=30, reports_dir='../reports')
print(metrics_df.to_string(index=False))

## 2. 30-Day Multi-Step Prediction with Confidence Spread

In [ ]:
rf_model = joblib.load('../models/rf_forecast.joblib')
h_id = df_hourly['household_id'].iloc[0]
h_data = df_hourly[df_hourly['household_id'] == h_id].tail(30 * 24)
forecast_30d = predict(rf_model, h_data, horizon_days=30)

plt.figure(figsize=(14, 5))
plt.plot(forecast_30d['timestamp'], forecast_30d['predicted_kwh'], label='RF Forecast (Mean)', color='#2ca02c')
plt.fill_between(forecast_30d['timestamp'], forecast_30d['lower_bound'], forecast_30d['upper_bound'], color='#2ca02c', alpha=0.2, label='10th - 90th Percentile Confidence Spread')
plt.title('WattWise AI: 30-Day Ahead Electricity Forecast with Uncertainty Bounds', fontsize=13, fontweight='bold')
plt.xlabel('Timestamp')
plt.ylabel('Predicted Hourly kWh')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Bill Estimation with Published UK Tariffs

In [ ]:
bill = estimate_bill(forecast_30d, tariff_per_kwh=0.28, standing_charge_per_day=0.60)
for k, v in bill.items():
    print(f'{k}: {v}')